
# **BioPred Phase 0 -- ChEMBL Discovery**

This notebook explores the local ChEMBL database for BioPred.

The goal is to identify the tables, columns, joins, and target records needed to design a reproducible GABA-A activity evidence extraction workflow.

This notebook is exploratory.  Final extraction and validation logic will be promoted later into reusable SQL or pipeline code.

## Target-family rationale

BioPred focuses on the GABA-A receptor family as a bounded CNS-relevant target family for early hit-prioritization modeling.  The goal is not to make therapeutic or clinical claims, but to build a focused discovery ML workflow where target scope, endpoint evidence, scaffold-aware validation, and ranked compound prioritization can be evaluated coherently.

This notebook uses ChEMBL to inspect whether sufficient GABA-A activity evidence exists to support downstream modeling.

## Notebook objectives:

- Confirm database access
- Identify relevant ChEMBL tables
- Inspect core table columns
- Search for GABA / GABA-A target candidates
- Check endpoint validity
- Identify early data-quality risks
- Draft the future evidence-table shape





In [1]:
# list imports needed for this notebook
from pathlib import Path
import sys
import os
from dotenv import load_dotenv
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# Force notebook runtime to project root
%cd /home/ryanm/projects/BioPred

# define paths and link src.paths
SRC_DIR = Path.cwd() / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import biopred.paths as paths

load_dotenv(paths.PROJECT_ROOT / ".env", override=True)

# create the database engine
db_url = URL.create(
    drivername = os.getenv("BIOPRED_DB_DIALECT"),
    username=os.getenv("BIOPRED_DB_USER"),
    password=os.getenv("BIOPRED_DB_PASSWORD"),
    host=os.getenv("BIOPRED_DB_HOST"),
    port=int(os.getenv("BIOPRED_DB_PORT")),
    database=os.getenv("BIOPRED_DB_NAME")
)

engine = create_engine(db_url, pool_pre_ping=True)
schema = os.getenv("BIOPRED_DB_SCHEMA", "public")


/home/ryanm/projects/BioPred


### **Section 1: Test the connection to the database**

Here we will make a simple connection to the database and run a test query to ensure everything is set up correctly.

In [2]:
# make a connection to the database and test
with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database(), current_schema();"))
    print(result.fetchone())

('chembl36', 'public')


### **Section 2: Identify Relevant ChEMBL Tables**

This section inspects the restored ChEMBL database to identify tables likely needed for BioPred's activity evidence extraction.  The goal is not to finalize joins yet, but to map the tables that contain activity values, assay metadata, target metadata, molecule identifiers, and compound structures.

In [3]:
# list all tables in our database
tables = pd.read_sql_query(
    text("""
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_schema = :schema
    ORDER BY table_name;
    """),
    con=engine,
    params={"schema": schema},
)

tables.head(30)

,table_schema,table_name
0,public,action_type
1,public,activities
2,public,activity_properties
3,public,activity_smid
4,public,activity_stds_lookup
5,public,activity_supp
6,public,activity_supp_map
7,public,assay_class_map
8,public,assay_classification
9,public,assay_parameters


In [4]:
# get the shape of the tables dataframe
tables.shape

(73, 2)

In [5]:
# using keywords to find tables of interest.  not final list, just a starting point to find tables related to drug discovery
keywords = [
    "activ",
    "assay",
    "target",
    "molecule",
    "compound",
    "structure",
    "component",
    "sequence",
    "doc",
    "source"
]

# find tables that contain any of the keywords in their name
candidate_tables = tables[
    tables["table_name"].str.contains("|".join(keywords), case=False, regex=True)
]

candidate_tables.head(35)

#candidate_tables.shape

,table_schema,table_name
1,public,activities
2,public,activity_properties
3,public,activity_smid
4,public,activity_stds_lookup
5,public,activity_supp
6,public,activity_supp_map
7,public,assay_class_map
8,public,assay_classification
9,public,assay_parameters
10,public,assay_type


### Candidate table search observations

The keyword-based search returned 35 candidate tables.  This is expected because the search was intentionally broad and designed for recall rather than precision.

At this stage, these tables should be treated as an inspection set, not the final BioPred ingestion table set.  The next step is to inspect column names and row counts to determine which tables are required for activity evidence extraction.

### **Section 3: Inspect Core Table Columns**

In this section we will look further into our list of tables from the ChEMBL database and look into their contents, seeing what would be useful for further application.

In [6]:
# make a helper function to examine the columns of a table, displaying as a df.
def inspect_columns(table_name: str) -> pd.DataFrame:
    """
    Return column metadata for a table in the active db schema.
    
    Parameters
    ----------
    table_name : str
        The name of the table to inspect.
    
    Returns
    -------
    pd.DataFrame
        Column names, data types, nullability, and ordinal position.   
    """  
    
    query = text("""
    SELECT
        table_schema,
        table_name,
        ordinal_position,
        column_name,
        data_type,
        is_nullable
    FROM information_schema.columns
    WHERE table_schema = :schema
      AND table_name = :table_name
    ORDER BY ordinal_position;
    """)
    
    return pd.read_sql_query(
        query,
        con=engine,
        params={
            "schema": schema,
            "table_name": table_name
        },
    )


In [7]:
# use on a table of interest to examine the columns
inspect_columns("activities")

,table_schema,table_name,ordinal_position,column_name,data_type,is_nullable
0,public,activities,1,activity_id,bigint,NO
1,public,activities,2,assay_id,bigint,NO
2,public,activities,3,doc_id,bigint,YES
3,public,activities,4,record_id,bigint,NO
4,public,activities,5,molregno,bigint,YES
5,public,activities,6,standard_relation,character varying,YES
6,public,activities,7,standard_value,numeric,YES
7,public,activities,8,standard_units,character varying,YES
8,public,activities,9,standard_flag,smallint,YES
9,public,activities,10,standard_type,character varying,YES


In [8]:
# use on a few more tables of interest to examine the columns.  I will inspect all 35 tables in the candidate_tables df, but here I will just show a few examples.
inspect_columns("activity_properties")

,table_schema,table_name,ordinal_position,column_name,data_type,is_nullable
0,public,activity_properties,1,ap_id,bigint,NO
1,public,activity_properties,2,activity_id,bigint,NO
2,public,activity_properties,3,type,character varying,NO
3,public,activity_properties,4,relation,character varying,YES
4,public,activity_properties,5,value,numeric,YES
5,public,activity_properties,6,units,character varying,YES
6,public,activity_properties,7,text_value,character varying,YES
7,public,activity_properties,8,standard_type,character varying,YES
8,public,activity_properties,9,standard_relation,character varying,YES
9,public,activity_properties,10,standard_value,numeric,YES


In [9]:
inspect_columns("target_dictionary")

,table_schema,table_name,ordinal_position,column_name,data_type,is_nullable
0,public,target_dictionary,1,tid,bigint,NO
1,public,target_dictionary,2,target_type,character varying,YES
2,public,target_dictionary,3,pref_name,character varying,NO
3,public,target_dictionary,4,tax_id,bigint,YES
4,public,target_dictionary,5,organism,character varying,YES
5,public,target_dictionary,6,chembl_id,character varying,NO
6,public,target_dictionary,7,species_group_flag,smallint,NO


In [10]:
inspect_columns("molecule_dictionary")

,table_schema,table_name,ordinal_position,column_name,data_type,is_nullable
0,public,molecule_dictionary,1,molregno,bigint,NO
1,public,molecule_dictionary,2,pref_name,character varying,YES
2,public,molecule_dictionary,3,chembl_id,character varying,NO
3,public,molecule_dictionary,4,max_phase,numeric,YES
4,public,molecule_dictionary,5,therapeutic_flag,smallint,NO
5,public,molecule_dictionary,6,dosed_ingredient,smallint,NO
6,public,molecule_dictionary,7,structure_type,character varying,NO
7,public,molecule_dictionary,8,molecule_type,character varying,YES
8,public,molecule_dictionary,9,first_approval,integer,YES
9,public,molecule_dictionary,10,oral,smallint,NO


### Initial candidate table classification

The keyword-based table search returned 35 candidate tables.  The goal is to identify the minimum table set needed for BioPred’s first activity-level evidence extraction, while deferring auxiliary tables until they solve a specific data, provenance, or validation need.  Here is a classification chart based on each table:

| Table | Apparent role | Provisional decision | Rationale |
|---|---|---|---|
| `activities` | Primary activity measurement records | Core now | Likely contains endpoint type, activity value, units, relation, pChEMBL value, assay key, and molecule key. This should anchor the evidence table. |
| `activity_properties` | Auxiliary metadata for activity records | Revisit later | May provide extra activity-level annotations, but likely not required if `activities` contains the primary measurement fields. Avoid joining early unless it solves a specific gap. |
| `activity_smid` | Activity standardization / mapping metadata | Revisit later | Potentially relevant to how activity records were standardized, but not needed for first extraction unless standardization provenance is unclear. |
| `activity_stds_lookup` | Lookup table for standardized activity fields | Revisit later | Could help interpret standardized endpoint/value fields, but first confirm `activities` already exposes usable standard fields. |
| `activity_supp` | Supplementary activity data | Revisit later | May contain supporting activity annotations. Defer to avoid unnecessary row expansion unless core activity fields are insufficient. |
| `activity_supp_map` | Mapping table for supplementary activity data | Revisit later | Likely supports `activity_supp`. Not part of the minimum evidence extraction path. |
| `assay_class_map` | Mapping between assays and assay classes | Revisit later | May help classify assays, but not required for the first activity/assay/target join. |
| `assay_classification` | Assay classification hierarchy/labels | Revisit later | Could support assay grouping later, but first-pass assay metadata likely comes from `assays`. |
| `assay_parameters` | Additional assay parameter metadata | Revisit later | Potentially useful for detailed assay context, but likely one-to-many and not needed for first extraction. |
| `assay_type` | Lookup table for assay type codes | Revisit later | Useful if assay type codes need decoding. First inspect whether `assays.assay_type` is already interpretable. |
| `assays` | Assay metadata and target linkage | Core now | Likely connects `activities` to targets through `tid` and contains assay type, description, confidence score, and document linkage. |
| `bio_component_sequences` | Biological component sequence metadata | Defer for now | Likely more relevant to biologics/biotherapeutics than small-molecule activity evidence. Revisit only if target/component mapping requires it. |
| `bioassay_ontology` | Ontology annotations for assays | Revisit later | Could enrich assay interpretation, but not needed for the first evidence table. |
| `biotherapeutic_components` | Biotherapeutic component metadata | Defer for v1 | BioPred v1 is focused on small-molecule hit prioritization from ChEMBL activity records; likely not needed. |
| `component_class` | Protein/component classification | Revisit later | May help with target family/subtype interpretation after core GABA-A targets are identified. |
| `component_domains` | Protein domain annotations | Defer for now | Could support deeper target biology, but unnecessary for initial activity evidence extraction. |
| `component_go` | Gene Ontology annotations for components | Defer for now | Potentially useful for biological annotation, but outside first-pass extraction. |
| `component_sequences` | Protein/component sequence metadata | Revisit later | May support GABA-A subunit/component interpretation after target discovery. Not required for first activity evidence table unless target metadata is insufficient. |
| `component_synonyms` | Synonyms for protein components | Revisit later | Could help identify GABA-A subunits or aliases, but should not be joined until target inclusion logic requires it. |
| `compound_properties` | Computed compound properties | Revisit later | May provide molecular descriptors, but BioPred should derive core features reproducibly with RDKit later. Do not rely on this initially. |
| `compound_records` | Compound-source/document records | Revisit later | May support source provenance or molecule-document relationships. Not needed before core activity extraction is validated. |
| `compound_structural_alerts` | Structural alert annotations | Defer for v1 | Could relate to safety/reactivity filters, but ADMET/toxicity-style filtering is outside BioPred v1 scope. |
| `compound_structures` | Chemical structure records | Core now | Needed for canonical SMILES and later RDKit featurization. Joins to molecule/activity records through `molregno`. |
| `docs` | Publication/source metadata | Revisit soon | Useful for provenance. May become part of the evidence table if `assays` or `activities` link cleanly through `doc_id`. |
| `molecule_atc_classification` | ATC drug classification metadata | Defer for now | May help annotate known drugs later, but not needed for first evidence extraction. Avoid using as a model feature. |
| `molecule_dictionary` | Molecule identifiers and metadata | Core now | Needed for ChEMBL molecule IDs and molecule-level metadata. Likely joins to `activities` and `compound_structures` through `molregno`. |
| `molecule_hierarchy` | Parent/salt/prodrug molecule relationships | Revisit later | May matter for molecule aggregation and duplicate handling, but not needed before basic evidence extraction. |
| `molecule_synonyms` | Molecule synonyms/names | Revisit later | Useful for rediscovery labels or display names, but not core for first extraction. |
| `site_components` | Binding/target site component mapping | Defer for now | Potentially relevant to detailed target-site biology, but outside the first GABA-A evidence extraction. |
| `source` | Source database/provider metadata | Revisit later | May support provenance if source fields are needed beyond `docs`. Not core until document/source joins are examined. |
| `target_components` | Target-to-component mapping | Revisit soon | Potentially important for GABA-A subunit/component interpretation. Not necessarily needed for first activity table, but likely useful for target inclusion validation. |
| `target_dictionary` | Target identity and metadata | Core now | Needed to identify GABA-A targets, organisms, target types, and ChEMBL target IDs. |
| `target_relations` | Relationships among targets | Revisit later | May help understand family/complex relationships, but defer until basic target discovery is complete. |
| `target_type` | Lookup table for target type codes | Revisit later | Useful if `target_dictionary.target_type` requires decoding or validation. |
| `variant_sequences` | Variant protein sequence metadata | Defer for now | Likely not needed for first-pass GABA-A hit prioritization unless variant-specific target records appear. |

##### Note this table classification is provisional.  It is based on table names and initial column metadata, and is intended only to define the minimum inspection set for BioPred's first activity-level evidence extraction.  Core tables will be validated further using row counts, sample records, and join checks before final extraction logic is written.

### **Section 4:  Search for GABA / GABA-A Target Candidates**

In this section we will look specifically at our target_dictionary table to see what values it has for GABA / GABA-A for our target, and what that will begin to look like for our data.

In [11]:
# As a precursor (and sanity check) to the next steps, I will also run a quick check on target_dictionary as it will be important for this section.
pd.read_sql_query(
    text("SELECT COUNT(*) AS n_targets FROM target_dictionary;"),
    con=engine
)

,n_targets
0,17803


Of these targets we will filter for GABA and GABA-A (noted below as longform gaba-aminobutyric) to see which targets are ideal for further analysis.

In [12]:
gaba_targets = pd.read_sql_query(
    text("""
    SELECT
        tid,
        chembl_id AS target_chembl_id,
        pref_name,
        organism,
        target_type
    FROM target_dictionary
    WHERE LOWER(pref_name) LIKE '%gaba%'
    OR LOWER(pref_name) LIKE '%gaba-aminobutyric%'
    ORDER BY organism, pref_name;
    """),
    con=engine,  
)

gaba_targets

,tid,target_chembl_id,pref_name,organism,target_type
0,120117,CHEMBL4680049,GABA-A receptor alpha-1/beta-1,Bos taurus,PROTEIN COMPLEX
1,104689,CHEMBL2094107,GABA-A receptor; anion channel,Bos taurus,PROTEIN COMPLEX GROUP
2,121125,CHEMBL5303569,GABA-A receptor; anion channel,Canis lupus familiaris,PROTEIN COMPLEX GROUP
3,105087,CHEMBL2111392,GABA A receptor alpha-1/beta-1/gamma-2,Homo sapiens,PROTEIN COMPLEX
4,104921,CHEMBL2111413,GABA A receptor alpha-2/beta-2/gamma-2,Homo sapiens,PROTEIN COMPLEX
5,105038,CHEMBL2111339,GABA A receptor alpha-3/beta-2/gamma-2,Homo sapiens,PROTEIN COMPLEX
6,105121,CHEMBL2111366,GABA A receptor alpha-4/beta-3/gamma-2,Homo sapiens,PROTEIN COMPLEX
7,105059,CHEMBL2111370,GABA A receptor alpha-6/beta-2/gamma-2,Homo sapiens,PROTEIN COMPLEX
8,117721,CHEMBL4106151,GABA-A receptor alpha-1/beta-3,Homo sapiens,PROTEIN COMPLEX
9,117727,CHEMBL4106157,GABA-A receptor alpha-4/beta-3/delta,Homo sapiens,PROTEIN COMPLEX


#### Initial GABA target classification

The broad target search returned 45 GABA-related ChEMBL targets.  This result includes in-scope human GABA receptor records as well as expected false positives such as GABA-B receptors, GABA-C receptors, GABA transporters, non-human targets, and ambiguous broad target records.

We would only be interested for the scope of this project in Human/Homo Sapien-related targets, which would reduce the target range of the output.

This classification again is provisional.  Final inclusion requires downstream assay/activity availability checks and endpoint counts for IC50, EC50, and Ki.

In [13]:
# Make a human-only filtered list of GABA-related targets for later use.
human_gaba_targets = gaba_targets[
    gaba_targets["organism"].eq("Homo sapiens")
].copy()

human_gaba_targets

,tid,target_chembl_id,pref_name,organism,target_type
3,105087,CHEMBL2111392,GABA A receptor alpha-1/beta-1/gamma-2,Homo sapiens,PROTEIN COMPLEX
4,104921,CHEMBL2111413,GABA A receptor alpha-2/beta-2/gamma-2,Homo sapiens,PROTEIN COMPLEX
5,105038,CHEMBL2111339,GABA A receptor alpha-3/beta-2/gamma-2,Homo sapiens,PROTEIN COMPLEX
6,105121,CHEMBL2111366,GABA A receptor alpha-4/beta-3/gamma-2,Homo sapiens,PROTEIN COMPLEX
7,105059,CHEMBL2111370,GABA A receptor alpha-6/beta-2/gamma-2,Homo sapiens,PROTEIN COMPLEX
8,117721,CHEMBL4106151,GABA-A receptor alpha-1/beta-3,Homo sapiens,PROTEIN COMPLEX
9,117727,CHEMBL4106157,GABA-A receptor alpha-4/beta-3/delta,Homo sapiens,PROTEIN COMPLEX
10,104291,CHEMBL1907597,GABA-A receptor; GABA-A site (alpha1/beta2 int...,Homo sapiens,PROTEIN COMPLEX
11,104905,CHEMBL2109244,GABA-A receptor; agonist GABA site,Homo sapiens,PROTEIN COMPLEX GROUP
12,104757,CHEMBL2095172,GABA-A receptor; alpha-1/beta-2/gamma-2,Homo sapiens,PROTEIN COMPLEX


In [14]:
# Filtering down again for only GABA-A targets and those related, making some exclusion rules.
exclude_mask = (
    human_gaba_targets["pref_name"].str.contains("GABA-B", case=False, na=False)
    | human_gaba_targets["pref_name"].str.contains("GABA-C", case=False, na=False)
    | human_gaba_targets["pref_name"].str.contains("transporter", case=False, na=False)
)

provisional_gabaa_targets = human_gaba_targets.loc[~exclude_mask].copy()
excluded_gaba_targets = human_gaba_targets.loc[exclude_mask].copy()

print(len(human_gaba_targets))
print(len(provisional_gabaa_targets))
print(len(excluded_gaba_targets))


22
17
5


There looks to be 17 target candidates from our provisional_gabaa_targets list that we have narrowed down for our GABAA target.  

### **Section 5: Check Endpoint Validity**

In order to further establish which specific target we should move forward with, we will check activity records for our provisional target list we have acquired.

Our purpose here is to answer if these targets actually have usable ChEMBL activity records for IC50, EC50, and Ki, in interpretable units, and tied to compounds.

We will just be auditing target-level and endpoint-level coverage before building the eventual ingestion query.

In [15]:
# Make a list of target_ids from our target list created previously
provisional_target_ids = provisional_gabaa_targets["target_chembl_id"].tolist()

len(provisional_target_ids), provisional_target_ids

(17,
 ['CHEMBL2111392',
  'CHEMBL2111413',
  'CHEMBL2111339',
  'CHEMBL2111366',
  'CHEMBL2111370',
  'CHEMBL4106151',
  'CHEMBL4106157',
  'CHEMBL1907597',
  'CHEMBL2109244',
  'CHEMBL2095172',
  'CHEMBL2094121',
  'CHEMBL2094130',
  'CHEMBL2094120',
  'CHEMBL2094122',
  'CHEMBL2095190',
  'CHEMBL2093872',
  'CHEMBL2109243'])

In [16]:
# sql query to show endpoint counts by target
endpoint_counts_sql = """
SELECT
    td.chembl_id AS target_chembl_id,
    td.pref_name,
    act.standard_type,
    COUNT(*) AS n_activity_rows,
    COUNT(DISTINCT act.molregno) AS n_unique_molecules
FROM target_dictionary td
JOIN assays a
    ON td.tid = a.tid
JOIN activities act
    ON a.assay_id = act.assay_id
WHERE td.chembl_id = ANY(%(target_ids)s)
    AND act.standard_type IN ('IC50', 'EC50', 'Ki')
GROUP BY
    td.chembl_id,
    td.pref_name,
    td.target_type,
    act.standard_type
ORDER BY
    td.pref_name,
    act.standard_type;
"""

endpoint_counts = pd.read_sql_query(
    endpoint_counts_sql,
    con=engine,
    params={
        "target_ids": provisional_target_ids},
)

endpoint_counts

,target_chembl_id,pref_name,standard_type,n_activity_rows,n_unique_molecules
0,CHEMBL2111392,GABA A receptor alpha-1/beta-1/gamma-2,EC50,24,15
1,CHEMBL2111392,GABA A receptor alpha-1/beta-1/gamma-2,IC50,4,4
2,CHEMBL2111392,GABA A receptor alpha-1/beta-1/gamma-2,Ki,6,6
3,CHEMBL2111413,GABA A receptor alpha-2/beta-2/gamma-2,EC50,55,44
4,CHEMBL2111413,GABA A receptor alpha-2/beta-2/gamma-2,IC50,8,8
5,CHEMBL2111413,GABA A receptor alpha-2/beta-2/gamma-2,Ki,82,81
6,CHEMBL2111339,GABA A receptor alpha-3/beta-2/gamma-2,EC50,22,11
7,CHEMBL2111339,GABA A receptor alpha-3/beta-2/gamma-2,IC50,4,4
8,CHEMBL2111339,GABA A receptor alpha-3/beta-2/gamma-2,Ki,55,51
9,CHEMBL2111366,GABA A receptor alpha-4/beta-3/gamma-2,EC50,9,9


The provisional human GABA-A target set has IC50 / EC50 / Ki coverage, but coverage is uneven and appears Ki-heavy.  

In [17]:
# Run a couple summaries from the output above.
endpoint_type_summary = (
    endpoint_counts
    .groupby("standard_type", as_index = False)
    [["n_activity_rows", "n_unique_molecules"]]
    .sum()
    .sort_values("n_activity_rows", ascending = False)
)

endpoint_type_summary

,standard_type,n_activity_rows,n_unique_molecules
2,Ki,4321,3459
0,EC50,677,513
1,IC50,581,470


The endpoint coverage audit shows that the provisional human GABA-A target set contains usable IC50, EC50, and Ki records, but the distribution is strongly endpoint-skewed. Ki records dominate the available activity data, while IC50 and EC50 provide smaller endpoint-specific subsets.

This supports carrying all three endpoint types forward into ingestion design while preserving endpoint identity as metadata. Later phases should evaluate endpoint-specific slices before deciding whether unified endpoint modeling, endpoint weighting, or endpoint-specific exclusions are appropriate. 

In [18]:
# One more summary table here
target_endpoint_summary = (
    endpoint_counts
    .groupby(["target_chembl_id", "pref_name"], as_index = False)
    [["n_activity_rows", "n_unique_molecules"]]
    .sum()
    .sort_values("n_activity_rows", ascending = False)
)

target_endpoint_summary

,target_chembl_id,pref_name,n_activity_rows,n_unique_molecules
3,CHEMBL2094121,GABA-A receptor; alpha-1/beta-3/gamma-2,966,744
4,CHEMBL2094122,GABA-A receptor; alpha-5/beta-3/gamma-2,952,781
2,CHEMBL2094120,GABA-A receptor; alpha-3/beta-3/gamma-2,878,713
5,CHEMBL2094130,GABA-A receptor; alpha-2/beta-3/gamma-2,768,601
1,CHEMBL2093872,GABA-A receptor; anion channel,676,579
6,CHEMBL2095172,GABA-A receptor; alpha-1/beta-2/gamma-2,505,378
7,CHEMBL2095190,GABA-A receptor; alpha-6/beta-3/gamma-2,336,227
13,CHEMBL2111413,GABA A receptor alpha-2/beta-2/gamma-2,145,133
10,CHEMBL2111366,GABA A receptor alpha-4/beta-3/gamma-2,86,81
8,CHEMBL2109244,GABA-A receptor; agonist GABA site,83,49


Endpoint coverage is uneven across the provisional human GABA-A target set.  The strongest coverage is concentrated in several receptor-complex records, especially:

- alpha-1/beta-3/gamma-2
- alpha-5/beta-3/gamma-2
- alpha-3/beta-3/gamma-2
- alpha-2/beta-3/gamma-2

These top target records account for most of the available IC50, EC50, and Ki activity rows.

This suggests BioPred v1 should preserve target identity as metadata and treat later subtype/family analysis as descriptive rather than mechanistic.

In [19]:
# One more table to wrap up
endpoint_pivot = (
    endpoint_counts
    .pivot_table(
        index=["target_chembl_id", "pref_name"],
        columns = "standard_type",
        values = "n_unique_molecules",
        aggfunc = "sum",
        fill_value = 0,
    )
    .reset_index()
)

endpoint_pivot.columns.name = None

endpoint_pivot = endpoint_pivot[
    ["target_chembl_id", "pref_name", "IC50", "EC50", "Ki"]
]

endpoint_pivot

,target_chembl_id,pref_name,IC50,EC50,Ki
0,CHEMBL1907597,GABA-A receptor; GABA-A site (alpha1/beta2 int...,0,37,0
1,CHEMBL2093872,GABA-A receptor; anion channel,353,9,217
2,CHEMBL2094120,GABA-A receptor; alpha-3/beta-3/gamma-2,2,27,684
3,CHEMBL2094121,GABA-A receptor; alpha-1/beta-3/gamma-2,10,50,684
4,CHEMBL2094122,GABA-A receptor; alpha-5/beta-3/gamma-2,2,18,761
5,CHEMBL2094130,GABA-A receptor; alpha-2/beta-3/gamma-2,1,18,582
6,CHEMBL2095172,GABA-A receptor; alpha-1/beta-2/gamma-2,74,224,80
7,CHEMBL2095190,GABA-A receptor; alpha-6/beta-3/gamma-2,3,9,215
8,CHEMBL2109244,GABA-A receptor; agonist GABA site,5,40,4
9,CHEMBL2111339,GABA A receptor alpha-3/beta-2/gamma-2,4,11,51


In [20]:
print(os.getcwd())

/home/ryanm/projects/BioPred


A much cleaner representation of our findings here, and you can see the disparity in endpoint data between target_chembl_ids.

#### Section 5 Conclusions:

- Proceed with IC50 / EC50 / Ki as candidate endpoints.
- Expect Ki-heavy and target-skewed data.
- We will not prune low-coverage targets yet.
- Preserve endpoint and target identity.

In [22]:
# Before moving on, we will save key artifacts for future usage.
provisional_targets_path = paths.INTERIM_DATA_DIR / "provisional_gabaa_targets.csv"

provisional_gabaa_targets.to_csv(provisional_targets_path, index=False)

print(provisional_targets_path)
print(provisional_gabaa_targets.shape)

/home/ryanm/projects/BioPred/data/interim/provisional_gabaa_targets.csv
(17, 5)


### **Section 6: Identify Early Data-Quality Risks**

This section is more of a risk register, to acknowledge and assign/name these risks now so that we can be aware and be ready to audit them later.

We identify risks that could affect label validity, model evaluation, and downstream claims.

##### Risk 1:  Endpoint Imbalance

We just previously discussed this in the previous section, where we discussed the provisional target set being strongly Ki-dominated.  The risk here is if endpoints are collapsed naively, the final model may learn patterns associated with Ki-rich binding records rather than a balanced potency signal.

Later mitigation steps:

- preserve `standard_type`
- evaluate endpoint-specific slices
- avoid claiming assay equivalence
- consider endpoint weighting only after endpoint-sliced results

##### Risk 2: Target imbalance

Endpoint coverage is concentrated in a small number of GABA-A target records. The top target records, especially beta-3/gamma-2-containing receptor complexes and the broader anion-channel record, account for most of the available IC50/EC50/Ki activity rows.

Later mitigation:

- preserve `target_chembl_id`
- report target-level coverage
- use target/family breakdowns descriptively
- avoid unsupported claims of uniform GABA-A family coverage

##### Risk 3: Low-coverage or zero-coverage provisional targets

Several provisional GABA-A target records have very limited IC50/EC50/Ki coverage, and some provisional targets did not appear in the endpoint audit. These records may contribute little to model training after strict activity cleaning.

Later mitigation:

- flag low-coverage targets during ingestion
- allow zero-coverage targets to fall out naturally
- avoid target-specific model claims for sparse targets
- keep low-coverage status visible in discovery documentation

##### Risk 4: Repeated measurements and duplicate molecule records

Activity row counts exceed unique molecule counts, indicating repeated measurements across assays, targets, endpoints, or documents. If duplicates are not handled carefully, frequently measured compounds or assay series could disproportionately influence labels and evaluation.

Later mitigation:

- define compound-target-endpoint aggregation rules
- inspect repeated measurements
- retain provenance fields
- avoid leakage across train/test from duplicated molecules or close analog series

##### Risk 5: Endpoint value validity

The current endpoint audit checks endpoint availability but does not yet enforce final value-quality rules. Later ingestion must verify standardized units, non-null positive values, relation symbols, and extreme potency values before pX transformation or active/inactive labeling.

Later mitigation:

- require interpretable units such as nM where appropriate
- exclude or separately handle null/non-positive values
- define policy for inequality relations (`>`, `<`, `>=`, `<=`)
- audit extreme values before pX transformation

##### Risk 6: Label-threshold sensitivity

The final active/inactive label definition may be sensitive to endpoint type, target composition, and potency threshold. A single global threshold may create endpoint-specific or target-specific label imbalance.

Later mitigation:

- report prevalence under candidate thresholds
- audit active/inactive counts by endpoint type
- audit active/inactive counts by target
- avoid selecting thresholds only to optimize model metrics

##### Risk 7: Target granularity ambiguity

The provisional target set includes both specific receptor-complex records and broader site-level or complex-group records. These may represent different levels of biological specificity. If treated identically without metadata, the model may mix subtype-specific and broader receptor-site activity signals.

Later mitigation:

- preserve `target_type`, `target_chembl_id`, and `pref_name`
- separate target-level and site-level summaries
- treat family/subtype analysis as descriptive
- avoid mechanistic claims about subtype selectivity unless supported by endpoint-specific evidence



### **Section 7: Draft Next-Step Evidence Table**

Based on the target and endpoint discovery above, the next ingestion step should produce a long-format activity evidence table. At this stage, one row should represent one ChEMBL activity record linked to a molecule, assay, endpoint, and provisional BioPred target.

The table should preserve:
- target metadata: `target_chembl_id`, `pref_name`, `target_type`
- assay metadata: `assay_id`, `assay_type`, `confidence_score`
- activity metadata: `activity_id`, `standard_type`, `standard_relation`, `standard_value`, `standard_units`, `pchembl_value`
- molecule metadata: `molregno`, `molecule_chembl_id`, `canonical_smiles`
- provenance metadata: `doc_id` or source document identifiers when available

Final pX transformation, active/inactive labeling, endpoint weighting, duplicate handling, Murcko scaffolds, fingerprints, and model-ready aggregation are deferred to later notebooks/scripts.

### **Conclusions and Next Steps**

This notebook established the initial ChEMBL discovery foundation for BioPred v1. It identified the relevant schema path, searched broad GABA-related targets, refined the human GABA-A target set, checked endpoint availability, and recorded early data-quality risks.

The broad GABA search returned 45 target records. Filtering to `Homo sapiens` produced 22 human GABA-related records. Manual refinement retained 17 provisional human GABA-A receptor-related targets and excluded 5 out-of-scope records: one GABA-B receptor, one GABA-C receptor, and three GABA transporter records.

Endpoint coverage exists for `IC50`, `EC50`, and `Ki`, but the data is strongly imbalanced. Ki dominates the available activity records, while IC50 and EC50 are smaller. Target coverage is also uneven, with most activity records concentrated in a small number of receptor-complex or receptor-group records.

The main early data-quality risks are endpoint imbalance, target imbalance, low-coverage targets, repeated activity measurements, endpoint value-quality issues, label-threshold sensitivity, scaffold/analog-series concentration, and target granularity ambiguity.

These findings support preserving endpoint type, target identity, assay metadata, molecule identifiers, and provenance during ingestion. Final activity cleaning, pX transformation, labeling, endpoint weighting, duplicate handling, scaffold generation, feature engineering, and modeling are deferred to later stages.

The next step is to build a long-format ChEMBL activity evidence table at activity-record grain for the provisional BioPred target set.